# Evaluator-Optimizer — Oil & Gas HSE Safe Work Procedure Drafting

## Pattern
The evaluator-optimizer pattern uses **one LLM to generate output and a second LLM to evaluate and provide feedback**, iterating until a quality threshold is met. The generator improves its output based on structured critique rather than just re-prompting.

This is the right pattern when:
- Quality can be objectively assessed against defined criteria
- Iterative refinement produces meaningfully better results
- A single-pass prompt reliably misses edge cases or safety gaps

## O&G Use Case: HSE Safe Work Procedure Drafting
Before any non-routine task on a well site — hot work, confined space entry, pressure testing, chemical handling — a Safe Work Procedure (SWP) must be drafted that covers hazard identification, controls, emergency response, and regulatory compliance.

A first draft written quickly often misses:
- Specific PPE requirements for the hazard profile
- Permit-to-work cross-references (hot work, confined space, LOTO)
- Emergency response steps tailored to the location
- Regulatory references (OSHA PSM, API RP 500, NFPA 70E)

The evaluator scores the draft against HSE golden rules and returns structured feedback. The generator revises until the procedure meets the quality bar — just like a senior HSE advisor reviewing and redlining a junior engineer's draft.

In [ ]:
%pip install openai -q

In [ ]:
import sys
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    sys.path.insert(0, '/Workspace' + _nb_path.rsplit('/', 1)[0])
except Exception:
    sys.path.insert(0, '.')

from util import llm_call, extract_xml

# ── Evaluator-Optimizer loop ──────────────────────────────────────────────────

PASS_THRESHOLD = 80   # Score out of 100 required to accept the procedure
MAX_ITERATIONS = 4    # Safety cap on iterations


def parse_evaluation(xml_text: str) -> dict:
    """Parse evaluator XML output into score + structured feedback."""
    import re
    score_match = re.search(r'<score>(\d+)</score>', xml_text)
    score = int(score_match.group(1)) if score_match else 0

    gaps = []
    for block in re.findall(r'<gap>(.*?)</gap>', xml_text, re.DOTALL):
        g = {}
        for field in ['rule', 'issue', 'required_fix']:
            m = re.search(f'<{field}>(.*?)</{field}>', block, re.DOTALL)
            g[field] = m.group(1).strip() if m else ''
        if g.get('issue'):
            gaps.append(g)

    verdict_match = re.search(r'<verdict>(.*?)</verdict>', xml_text, re.DOTALL)
    verdict = verdict_match.group(1).strip() if verdict_match else ''

    return {'score': score, 'gaps': gaps, 'verdict': verdict}


def draft_swp(task_description: str, site_context: str,
               feedback: str = "") -> str:
    """Generator: draft or revise a Safe Work Procedure."""
    revision_instruction = ""
    if feedback:
        revision_instruction = f"""
A senior HSE advisor has reviewed your previous draft and identified the following gaps.
You MUST address every gap listed below in your revised procedure:

{feedback}

Revise the procedure below, ensuring all gaps above are explicitly resolved.
"""

    prompt = f"""You are an HSE engineer drafting a Safe Work Procedure (SWP) for an oil and gas well site.
{revision_instruction}
Task to be performed: {task_description}

Site context:
{site_context}

Write a complete Safe Work Procedure covering:
1. Scope and applicability
2. Hazard identification and risk assessment
3. Required permits (hot work, confined space, LOTO, cold work as applicable)
4. Personal protective equipment (PPE) — be specific to the hazard profile
5. Step-by-step work instructions with safety controls at each step
6. Emergency response procedures (spill, fire, injury, H2S release as applicable)
7. Regulatory references (OSHA, API RP, NFPA as applicable)
8. Sign-off requirements

Be specific and actionable. Vague instructions like 'wear appropriate PPE' are not acceptable."""

    return llm_call(prompt)


def evaluate_swp(swp_draft: str, task_description: str) -> dict:
    """Evaluator: score the SWP against HSE golden rules and return structured gaps."""
    prompt = f"""You are a senior HSE auditor reviewing a Safe Work Procedure for an oil and gas well site.

Task being performed: {task_description}

Evaluate the procedure below against these HSE golden rules:
1. PERMIT: All non-routine work requires a valid permit-to-work referenced in the procedure.
2. PPE: PPE must be specific (e.g. 'H2S monitor rated >20 ppm', not just 'gas detector').
3. LOTO: Any energy isolation must reference LOTO procedure with verification steps.
4. EMERGENCY: Emergency response must cover the top 3 hazards for the specific task/location.
5. REGULATORY: At least one specific regulatory standard must be cited (OSHA 1910.119, API RP 500, etc.).
6. STEPWISE: Work instructions must be sequential, numbered, and include a safety control at each step.
7. SIGN-OFF: Procedure must specify who reviews and approves before work commences.

Respond in XML:
<evaluation>
  <score>0-100 integer — overall quality score</score>
  <gaps>
    <gap>
      <rule>Which golden rule is violated or insufficiently addressed</rule>
      <issue>Specific text or omission in the draft that fails this rule</issue>
      <required_fix>Exactly what must be added or changed to pass</required_fix>
    </gap>
    ... (list every gap found, or empty if none)
  </gaps>
  <verdict>PASS (score >= 80) or REVISE with 1-sentence summary of the most critical gap</verdict>
</evaluation>

SWP Draft:
{swp_draft}"""

    response = llm_call(prompt)
    return parse_evaluation(response)


def run_evaluator_optimizer(task_description: str, site_context: str) -> dict:
    """Run the full evaluator-optimizer loop for a Safe Work Procedure.

    Args:
        task_description: What work is being performed.
        site_context: Location, equipment, hazard profile, and relevant site data.

    Returns:
        Dict with iteration history and the accepted final procedure.
    """
    print(f"\n{'='*60}")
    print(f"TASK: {task_description}")
    print('='*60)

    history = []
    feedback_text = ""
    final_swp = None

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n[Iteration {iteration}] Generating SWP draft...")
        draft = draft_swp(task_description, site_context, feedback_text)

        print(f"[Iteration {iteration}] Evaluating against HSE golden rules...")
        evaluation = evaluate_swp(draft, task_description)

        score = evaluation['score']
        gaps = evaluation['gaps']
        verdict = evaluation['verdict']

        print(f"[Iteration {iteration}] Score: {score}/100 | Gaps: {len(gaps)} | {verdict}")
        if gaps:
            for g in gaps:
                print(f"  • [{g['rule']}] {g['issue'][:80]}..." if len(g['issue']) > 80 else f"  • [{g['rule']}] {g['issue']}")

        history.append({
            'iteration': iteration,
            'score': score,
            'gaps': gaps,
            'verdict': verdict,
            'draft': draft
        })

        if score >= PASS_THRESHOLD:
            print(f"\n✓ Procedure ACCEPTED at iteration {iteration} (score {score}/100)")
            final_swp = draft
            break

        # Build feedback for next iteration
        feedback_text = "\n".join([
            f"GAP {i+1} — {g['rule']}:\n  Issue: {g['issue']}\n  Required fix: {g['required_fix']}"
            for i, g in enumerate(gaps)
        ])

    if final_swp is None:
        # Accept best scoring iteration if we hit the cap
        best = max(history, key=lambda h: h['score'])
        final_swp = best['draft']
        print(f"\n⚠ Max iterations reached. Best score: {best['score']}/100 (iteration {best['iteration']})")

    return {
        'task': task_description,
        'iterations': len(history),
        'scores': [h['score'] for h in history],
        'history': history,
        'final_swp': final_swp
    }

## Mock Well Site Tasks — Three HSE Scenarios
Each task has different hazard profiles. The evaluator should catch different gaps in each first draft.

In [ ]:
TASKS = [
    {
        "description": "Hot work (welding) on a 6-inch produced water transfer line",
        "context": """
        Site: Permian Basin Block 7 West — Well Pad C
        Location: Produced water transfer line, 50 ft from active separator vessels
        Line contents: Produced water with residual hydrocarbons (last pigged 6 months ago)
        H2S: Detected at 8 ppm near separator; wind NW at 12 mph
        Nearest muster point: 300 ft east of pad
        Work window: Daytime only, 0700-1700
        Regulatory jurisdiction: Texas (OSHA 1910.119 PSM site, > 10,000 lb flammable threshold)
        Emergency contacts: Ector County 911, site medic on call, company ERT
        """
    },
    {
        "description": "Confined space entry to inspect and clean a 500-barrel oil storage tank",
        "context": """
        Site: Permian Basin Block 7 West — Tank Battery A
        Tank: 500 bbl fixed-roof atmospheric storage tank, last cleaned 2 years ago
        Contents (drained): Residual crude sludge (~3 inch layer on floor)
        Atmosphere: Pre-entry monitor showed O2 19.8%, LEL 22%, H2S 4 ppm
        Entry hatch: 24-inch top manway, single entry point
        Nearest hospital: Midland Memorial, 45 min by road
        Regulatory jurisdiction: Federal OSHA 29 CFR 1910.146 (Permit-Required Confined Spaces)
        Entrants: Max 2 at a time; 1 attendant outside at all times
        """
    },
    {
        "description": "Pressure testing a newly installed 2-inch gas injection flowline to 3,000 psi",
        "context": """
        Site: Permian Basin Block 7 West — Well B-09 injection skid
        Test medium: Nitrogen (inert gas pressure test)
        MAOP: 2,500 psi; Test pressure: 3,000 psi (120% MAOP per API 1110)
        Line length: 450 ft, 2-inch schedule 80 carbon steel
        Fittings: 4 × flanged connections, 2 × threaded connections (potential failure points)
        Exclusion zone required: Per ASME B31.3 and company procedures
        Regulatory: API RP 1110, ASME B31.3, DOT 49 CFR Part 192
        Weather: Forecast 35°F overnight — temperature effect on test pressure must be addressed
        """
    }
]

print(f"{len(TASKS)} HSE tasks queued for Safe Work Procedure drafting")

## Run Evaluator-Optimizer Loop for Each Task

In [ ]:
all_results = []
for task in TASKS:
    result = run_evaluator_optimizer(task['description'], task['context'])
    all_results.append(result)

## Results — Iteration Scores and Final Accepted Procedures

In [ ]:
for r in all_results:
    print(f"\n{'='*60}")
    print(f"  {r['task'][:55]}")
    print('='*60)
    scores = r['scores']
    for i, s in enumerate(scores, 1):
        bar = '█' * (s // 5) + '░' * (20 - s // 5)
        status = '✓ ACCEPTED' if (i == len(scores) and s >= 80) else ('↺ revised' if i < len(scores) else '⚠ best effort')
        print(f"  Iteration {i}: [{bar}] {s}/100  {status}")
    print(f"\n--- ACCEPTED PROCEDURE ---")
    print(r['final_swp'])

## Why the Evaluator-Optimizer Matters for HSE

A single-pass LLM prompt can draft a plausible-looking SWP that still fails on specifics:

| Gap type | What a first draft often misses | What the evaluator catches |
|---|---|---|
| Permit specificity | "Obtain hot work permit" | Must reference PTW number format and issuing authority |
| PPE vagueness | "Wear appropriate PPE" | H2S monitor rated to specific ppm, FR coverall ATPV rating |
| LOTO steps | "Isolate the line" | Blind flange location, zero-energy verification steps |
| Emergency gaps | Generic evacuation | H2S release protocol with wind direction, muster point, headcount |
| Regulatory citations | None | OSHA 1910.119, API RP 500 section number |

The loop terminates when all gaps are resolved — typically 2-3 iterations — producing a procedure that would pass a field HSE audit.

**Where to go from here:**
- Load your company's specific HSE golden rules from Unity Catalog as the evaluation rubric
- Feed approved historical SWPs as few-shot examples to the generator
- Export accepted procedures directly to your work order management system (SAP PM, IBM Maximo)
- Run weekly across all planned non-routine tasks to pre-approve procedures before the work window